In [37]:
from scipy.stats import median_abs_deviation

import sys
sys.path.insert(0, '/home/workspace/mm_bm_chip_organoids')
sys.path.insert(0, '/home/workspace/')

from py_util import *
from utilities import *

hdir = '/home/workspace'
wdir = hdir + "/mm_bm_chip_organoids/EXP-01440"
objdir = wdir + "/object_building/objects/"

adata = sc.read_h5ad(objdir + "not_ds_all_organoids.h5ad")

# Not Downsampled

In [27]:
cr_outs_path = os.path.join(hdir, "mm_bm_chip_organoids/EXP-01440/cr_outs")
path = wdir + "/sample_reference.csv"
df = pd.read_csv(path)

sample_dict = {name: donor for name, donor in zip(df.iloc[:, 0], df.iloc[:, 4])}

type_dict = {name: sample_type for name, sample_type in zip(df.iloc[:, 0], df.iloc[:, 2])}

h5_paths = [os.path.join(root, 'sample_filtered_feature_bc_matrix.h5') 
           for root, _, files in os.walk(cr_outs_path) 
           if 'sample_filtered_feature_bc_matrix.h5' in files]

final_adatas = {}

for sample in list(sample_dict.keys()):

    # Get only the sample h5 paths
    paths = [path for path in h5_paths if sample in path]

    # Dictionary to store AnnData objects for each sample
    adatas = {}
    
    # Process each H5 file
    for path in paths:
        # Extract sample name from path
        name = path.split('per_sample_outs/')[1].split('/')[0]
        
        # Read the H5 file and create AnnData object
        adata = sc.read_10x_h5(path)
        adata.var_names_make_unique()
        
        adata.obs['sample'] = name
        adata.obs['donor'] = adata.obs['sample'].replace(sample_dict)
        adata.obs['type'] = adata.obs['sample'].replace(type_dict)
        
        adatas[name] = adata.copy()
    
    adata = ad.concat(adatas.values(), join='outer', merge='same')
    
    final_adatas[adata.obs['sample'].unique()[0]] = adata.copy()

adata = ad.concat(final_adatas.values(), join = 'outer', merge = 'same')

print(adata.obs['sample'].unique())

adata.write(objdir + 'not_ds_all_organoids.h5ad', compression='gzip')

['BMC07965-007' 'BMC09023-003' 'BMC09025-003' 'CELL01173' 'CELL01174'
 'CELL01176' 'OR00005' 'OR00009' 'OR00013' 'OR00017' 'OR00019' 'OR00023'
 'OR00029' 'OR00031' 'OR00035']


In [36]:
names = list(adata.obs['sample'].unique())

stats = {
    'id': [name for name in names],
    'total cells': [adata[adata.obs['sample'] == name].n_obs for name in names]
}

stats_df = pd.DataFrame(stats)

stats_df

,id,total cells
0,BMC07965-007,2366
1,BMC09023-003,1369
2,BMC09025-003,941
3,CELL01173,11943
4,CELL01174,4470
5,CELL01176,5017
6,OR00005,8365
7,OR00009,27436
8,OR00013,16347
9,OR00017,428
